## **Exploratory Data Analysis**

#### **Dataset**

We decided to go on the adversarial QA dataset as it a very interesting one to evaluate the flan-T5 model, how it performs initially on an adversial dataset without fine-tuning and on different quantisation level, to see how quantisation affect performances. Then later on we will fine tune the model on the dataset in order to see how fine tuning affect the performance of the model on the different quantisation levels. 

In [31]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer

In [64]:
model_id = "google/flan-t5-base"
ds = load_dataset("UCLNLP/adversarial_qa", "adversarialQA")
device = "cuda:0" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(device)

mps


Let us explore the data set

In [4]:
ds

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers', 'metadata'],
        num_rows: 30000
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers', 'metadata'],
        num_rows: 3000
    })
    test: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers', 'metadata'],
        num_rows: 3000
    })
})

In [73]:
for set in ds:
    n = len(ds[set])
    n_empty = sum(1 for a in ds[set]["answers"] if len(a["text"]) == 0)
    print(f"{set:12s} {n:6d} rows | {n_empty:5d} rows without answer")

train         30000 rows |     0 rows without answer
validation     3000 rows |     0 rows without answer
test           3000 rows |  3000 rows without answer


As we can see the dataset is already split into train, validation and test sets. As mentioned in the documentation, https://huggingface.co/datasets/UCLNLP/adversarial_qa, there are no answer provided with the test set. We'll therefore ignore it, use the current validation set as our test set, and modify the train set to include 27k examples instead of 30k, keeping the 3k for a new validation set.

In [89]:
split = ds["train"].train_test_split(test_size=3000, seed=42)
train_ds = split["train"]
val_ds = split["test"]
test_ds = ds["validation"]
lengths = {"train":len(train_ds), "val":len(val_ds), "test":len(test_ds)}

print("Numbers of rows in each set:")
for set, length in lengths.items():
    print(f"{set:6s}: {length:6d} rows")

Numbers of rows in each set:
train :  27000 rows
val   :   3000 rows
test  :   3000 rows


Let us explore also the tokens length of the context + question that we will later use for the prompt of the model. We'll use for that the tokenizer of the model we chose for the project, which is flan-t5-large

In [ ]:
tokenizer = AutoTokenizer(model_id)


32100 32100
True
